# Gold Layer — dim_customer

Joins all customer-related silver tables (CRM + ERP) into a single `workspace.gold.dim_customer` dimension table, implemented as **SCD Type 2** to preserve the full change history of each customer.

---

## Table of Contents — Transformations Applied

| # | Transformation | Description |
|---|---|---|
| 1 | **Three-Way Join** | Joined CRM `customer_info` with ERP `customer_info1` (cust_az12) and ERP `customer_info2` (loc_a101) using SQL LEFT JOINs |
| 2 | **Direct Join** | All silver customer IDs are now in `AW000xxxxx` format, enabling direct equality joins on `customer_key` — no key transformation needed |
| 3 | **Unified Customer Dimension** | Combined CRM demographics (name, marital status, gender, create date) with ERP data (birth date, country) into a single dimension table |

---

## Source Tables

| Source Table | Layer | Columns Contributed |
|---|---|---|
| `workspace.silver.customer_info` | CRM | customer_id, customer_key, first_name, last_name, full_name, marital_status, gender, create_date |
| `workspace.silver.customer_info1` | ERP (cust_az12) | birth_date, gender (ERP source) |
| `workspace.silver.customer_info2` | ERP (loc_a101) | country |

---

## Join Logic

```
CRM customer_info (base, 18,494 rows)
  LEFT JOIN silver.customer_info2 (loc_a101)
    ON ci2.customer_id = ci.customer_key
  LEFT JOIN silver.customer_info1 (cust_az12)
    ON ci1.customer_id = ci.customer_key
```

| Metric | Count |
|---|---|
| Total Gold Rows | 18,494 |
| Matched with loc_a101 | 18,490 |
| Matched with cust_az12 | 18,490 |
| Unmatched (CRM only) | 4 |

---

## SCD Type 2 Design

| Column | Role |
|---|---|
| `customer_sk` | Surrogate primary key — unique across **all** versions |
| `customer_id` | Business key — unique among **current** rows only (`is_current = true`) |
| `start_date` / `end_date` | Version validity window (`end_date` IS NULL = active version) |
| `is_current` | Marks the active version of each customer |

Tracked attributes (a change in any of them opens a new version): `customer_key`, `first_name`, `last_name`, `full_name`, `birth_date`, `gender`, `marital_status`, `country`.

**Source deduplication (data quality):** `silver.customer_info` contains duplicate `customer_id` rows (progressively-filled records with different `create_date`) plus 4 junk rows with NULL `customer_id`. Every gold source query keeps ONE row per customer — the latest `create_date` — and excludes NULL ids, so `dim_customer` holds 18,484 rows (one current version per customer).

**Refresh steps:**

1. **Initial SCD2 load** — run once to (re)build the table with the SCD2 columns.
2. **Close outdated versions** — MERGE sets `is_current = false` and `end_date = today` on the current row of any customer whose tracked attributes changed (NULL-safe `<=>` comparisons).
3. **Insert new versions** — inserts a fresh current row for changed customers (new version) and for brand-new customers.

---

## Implementation

All joins are implemented using **SQL** with LEFT JOIN to preserve all CRM customer records.

**Downstream note:** `fact_sales` should join on `customer_sk`, or add `AND d.is_current = true` when joining on `customer_id` — otherwise changed customers would match multiple rows.

In [0]:
%sql
-- INITIAL SCD2 LOAD (run once): rebuilds dim_customer with SCD Type 2 columns.
-- customer_sk  : surrogate PK, unique across all versions
-- customer_id  : business key, unique among current rows only
-- start_date/end_date/is_current : SCD2 validity tracking
-- Source is deduplicated: keeps ONE row per customer_id (latest create_date), NULL customer_id rows excluded.
-- Re-running this cell WIPES history — for incremental refreshes use the two cells below.

CREATE OR REPLACE TABLE workspace.gold.dim_customer
COMMENT 'Gold-layer customer dimension (SCD Type 2). customer_sk is the surrogate PK; customer_id is the business key, unique among current rows.'
AS
SELECT
    CAST(ROW_NUMBER() OVER (ORDER BY customer_id) AS INT) AS customer_sk,
    customer_id,
    customer_key,
    first_name,
    last_name,
    full_name,
    birth_date,
    gender,
    marital_status,
    country,
    create_date,
    current_date()      AS start_date,
    CAST(NULL AS DATE) AS end_date,
    true               AS is_current
FROM (
    SELECT
        ci.customer_id,
        ci.customer_key,
        ci.first_name,
        ci.last_name,
        ci.full_name,
        ci1.birth_date,
        ci.gender,
        ci.marital_status,
        ci2.country,
        ci.create_date
    FROM workspace.silver.customer_info ci
    LEFT JOIN workspace.silver.customer_info2 ci2
        ON ci2.customer_id = ci.customer_key
    LEFT JOIN workspace.silver.customer_info1 ci1
        ON ci1.customer_id = ci.customer_key
    WHERE ci.customer_id IS NOT NULL
    QUALIFY ROW_NUMBER() OVER (PARTITION BY ci.customer_id ORDER BY ci.create_date DESC) = 1
)

In [0]:
%sql
-- INCREMENTAL REFRESH — STEP 1/2: close outdated current versions.
-- Compares the fresh silver snapshot against the CURRENT version of each customer.
-- If any tracked attribute changed (NULL-safe <=> comparison), the old version is
-- closed: is_current = false, end_date = today.

MERGE INTO workspace.gold.dim_customer AS d
USING (
    SELECT
        ci.customer_id,
        ci.customer_key,
        ci.first_name,
        ci.last_name,
        ci.full_name,
        ci1.birth_date,
        ci.gender,
        ci.marital_status,
        ci2.country
    FROM workspace.silver.customer_info ci
    LEFT JOIN workspace.silver.customer_info2 ci2
        ON ci2.customer_id = ci.customer_key
    LEFT JOIN workspace.silver.customer_info1 ci1
        ON ci1.customer_id = ci.customer_key
    WHERE ci.customer_id IS NOT NULL
    QUALIFY ROW_NUMBER() OVER (PARTITION BY ci.customer_id ORDER BY ci.create_date DESC) = 1
) AS s
    ON d.customer_id = s.customer_id
   AND d.is_current = true
WHEN MATCHED AND (
        NOT (d.customer_key    <=> s.customer_key)
     OR NOT (d.first_name     <=> s.first_name)
     OR NOT (d.last_name      <=> s.last_name)
     OR NOT (d.full_name      <=> s.full_name)
     OR NOT (d.birth_date     <=> s.birth_date)
     OR NOT (d.gender         <=> s.gender)
     OR NOT (d.marital_status <=> s.marital_status)
     OR NOT (d.country        <=> s.country)
) THEN UPDATE SET
    d.is_current = false,
    d.end_date   = current_date()

In [0]:
%sql
-- INCREMENTAL REFRESH — STEP 2/2: insert new current versions.
-- After step 1 closed the old rows, changed customers (and brand-new customers)
-- have no is_current row left. This inserts a fresh current version for them,
-- with a new customer_sk above the current MAX(customer_sk).

INSERT INTO workspace.gold.dim_customer
SELECT
    (SELECT COALESCE(MAX(customer_sk), 0) FROM workspace.gold.dim_customer)
        + CAST(ROW_NUMBER() OVER (ORDER BY s.customer_id) AS INT) AS customer_sk,
    s.customer_id,
    s.customer_key,
    s.first_name,
    s.last_name,
    s.full_name,
    s.birth_date,
    s.gender,
    s.marital_status,
    s.country,
    s.create_date,
    current_date()      AS start_date,
    CAST(NULL AS DATE) AS end_date,
    true               AS is_current
FROM (
    SELECT
        ci.customer_id,
        ci.customer_key,
        ci.first_name,
        ci.last_name,
        ci.full_name,
        ci1.birth_date,
        ci.gender,
        ci.marital_status,
        ci2.country,
        ci.create_date
    FROM workspace.silver.customer_info ci
    LEFT JOIN workspace.silver.customer_info2 ci2
        ON ci2.customer_id = ci.customer_key
    LEFT JOIN workspace.silver.customer_info1 ci1
        ON ci1.customer_id = ci.customer_key
    WHERE ci.customer_id IS NOT NULL
    QUALIFY ROW_NUMBER() OVER (PARTITION BY ci.customer_id ORDER BY ci.create_date DESC) = 1
) s
LEFT JOIN workspace.gold.dim_customer d
    ON d.customer_id = s.customer_id
   AND d.is_current = true
WHERE d.customer_id IS NULL

In [0]:
%sql
-- Verify SCD2 state:
--  one current row per customer, no active row with an end_date,
--  and every closed row having start_date <= end_date.

SELECT
    COUNT(*)                                              AS total_rows,
    COUNT(DISTINCT customer_id)                           AS distinct_customers,
    SUM(CASE WHEN is_current THEN 1 ELSE 0 END)          AS current_rows,
    COUNT(DISTINCT CASE WHEN is_current THEN customer_id END) AS distinct_current_customers,
    SUM(CASE WHEN is_current AND end_date IS NOT NULL THEN 1 ELSE 0 END) AS broken_current_rows,
    SUM(CASE WHEN NOT is_current AND (end_date IS NULL OR end_date < start_date) THEN 1 ELSE 0 END) AS broken_history_rows
FROM workspace.gold.dim_customer;

-- Show any customer with more than one version (history in action)
SELECT customer_id, COUNT(*) AS versions
FROM workspace.gold.dim_customer
GROUP BY customer_id
HAVING COUNT(*) > 1
LIMIT 10;

In [0]:
%sql
SELECT * FROM workspace.gold.dim_customer LIMIT 10